# Transformer

> 《Attention Is All You Need》NeurIPS 2017

![transformer](https://github.com/Larry0454/Technical-Learning-Notes/blob/main/AI/DL/pics/transformer.png)

---

## 一、Input Embeddings

In [ ]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

import warnings
warnings.filterwarnings("ignore")

class InputEmbedding(nn.Module):
    """
    输入的嵌入向量表, 将单词序号转化为嵌入向量
    d_model: 嵌入向量的维度
    vocab_size: 词表大小
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)
     
    """
    直接将单词id传入embedding_table即可, 形状: (batch, seq_len) -> (batch, seq_len, d_model)
    这里原文提出要额外乘以一个权重sqrt(d_model)
    """
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

---

## 二、Positional Embeddings

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    位置嵌入向量和单词嵌入向量维度相同, 都是d_model, 便于两者逐元素相加
    这里采用了一维位置编码, sinusoidal positional encoding, 属于**绝对位置编码**
    """
    def __init__(self, d_model: int, seq_len: int):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        assert d_model % 2 == 0, "d_model must be even"
        
        """
        构建pos_mat和i2_mat, 计算三角函数内的值
        pos_mat形状为(seq_len, 1)
        i2_mat形状为(1, model_dim/2)
            假设2*i=0,2,4,6
            则pe中的偶数列(2i)的嵌入值为2*i
            则pe中的奇数列(2i+1)也嵌入值为2*i
            (另一种求法参考**position_embedding节**)
        """
        pe = torch.zeros(seq_len, d_model)
        pos_mat = torch.arange(seq_len).reshape((-1, 1))
        i2_mat = torch.pow(10000, torch.arange(0, d_model, 2).reshape((1, -1)) / d_model)
        
        """
        偶数列用sin嵌入, 奇数列用cos嵌入
        """
        pe[:, 0::2] = torch.sin(pos_mat / i2_mat)
        pe[:, 1::2] = torch.cos(pos_mat / i2_mat)
        
        """
        支持批处理, 形状为(1, seq_len, d_model), 利用广播机制将**一个pe复制batch份**, 从而支持对批次内的所有句子进行位置编码
        将不可学习的张量保存在模型参数文件中, 需要将其注册为缓冲区
        """
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    """
    对输入批次x中的每个句子叠加位置嵌入, 形状为(batch, seq_len, d_model)
    手动修改pe的requires_grad属性为False, 表示不更新位置嵌入编码
    这里只截取x.shape[1]长度的部分, 因为推理过程中Decoder的Input序列是逐渐延长的
    """
    def forward(self, x):
        # input_embedding + positional_embedding
        x = x + (self.pe[:, x.shape[1], :]).requires_grad_(False)
        return x

---

## 三、FeedForwardBlock

In [ ]:
class FeedForwardBlock(nn.Module):
    """
    构造两层前向连接层FFN(x)
    FFN(x) = max(0, xW1 + b1)W2 + b2
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff) # W1, b1
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model) # W2, b2
        
    """
    (batch, seq_len, d_model) -> (batch, seq_len, d_ff) -> (batch, seq_len, d_model)
    """
    def forward(self, x):
        x = self.linear1(x)
        x = self.dropout(F.relu(x))
        x = self.linear2(x)
        return x

---

## 四、MultiHead Self-Attention

In [ ]:
class MultiHeadAttentionBlock(nn.Module):
    """
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k))V
    一个头负责访问批次内整个句子的一部分内容
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model) # Wq
        self.w_k = nn.Linear(d_model, d_model) # Wk
        self.w_v = nn.Linear(d_model, d_model) # Wv
        
        self.w_o = nn.Linear(d_model, d_model) # Wo
        self.dropout = nn.Dropout(dropout)
        
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        
        """
        (batch, num_heads, seq_len, d_k) @ (batch, num_heads, d_k, seq_len) -> (batch, num_heads, seq_len, seq_len)
        除以sqrt(d_k)是为了让注意力分数分布更加集中, 避免梯度消失
        对于推理阶段, attention_score是一个长方形, 形状可以为(batch, num_heads, 2, seq_len) (比如长方形)
        """
        attention_score = (query @ key.transpose(-2, -1) / math.sqrt(d_k))
        if mask is not None:
            """
            enc_mask形状为(batch, 1, 1, seq_len), dec_mask形状为(batch, 1, seq_len, seq_len)
            通过广播机制将mask广播到所有4个维度
            同时将mask为0的上三角(不含对角线)填为-1e9(即-inf), 经过softmax后将指定部分置0
            """
            attention_score = attention_score.masked_fill_(mask == 0, -1e9)
            
        attention_score = attention_score.softmax(dim=-1)  # (batch, num_heads, seq_len, seq_len), 两个seq_len可以不相等
        if dropout is not None:
            attention_score = dropout(attention_score)
           
        """
        最后与多头的V相乘
        (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, d_k) -> (batch, num_heads, seq_len, d_k)
        """ 
        return (attention_score @ value)
        
         
    def forward(self, q, k, v, mask):
        """
        利用输入的x, 分三路同时得到Q, K, V
        (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        """
        query = self.w_q(q)
        key   = self.w_k(k)
        value = self.w_v(v)
        
        """
        把Q, K, V分别拆分成多个头: (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
        保证一个head能看到批次内的所有句子(的一部分), 即所有的(seq_len, d_k)
        """
        query = query.view(query.shape[0], self.num_heads, query.shape[1], self.d_k)
        key   = key.view(key.shape[0], self.num_heads, key.shape[1], self.d_k)
        value = value.view(value.shape[0], self.num_heads, value.shape[1], self.d_k)
        
        """
        将按head拆分过的Q, K, V输入到多头注意力机制
        x形状为(batch_size, num_heads, seq_len, d_k), 与多头query, key, value的形状相同
        """
        x = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)
        
        """
        先把多个头重新concat起来: (batch, num_heads, seq_len, d_k) -> (batch, seq_len, num_heads * d_k) = (batch, seq_len, d_model)
        再乘以矩阵Wo: (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        """
        x = x.transpose(1, 2).reshape(x.shape[0], -1, self.num_heads * self.d_k)
        return self.w_o(x)

---

## 五、Residual Connection & LayerNormalization

In [ ]:
class LayerNormalization(nn.Module):
    def __init__(self, eps: float=10**-6):
        super().__init__()
        self.eps = eps
        """
        gamma和beta都是可学习的参数
        由nn.Parameter表示
        """
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        """
        层归一化求解**单个样本内部**各分量的均值和方差
        形状为 (batch, seq_len, d_model) - (batch, seq_len, 1) -广播-> (batch, seq_len, d_model)
        """
        mean = x.mean(dim=-1, keepdim=True)
        std  = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta # 分母加eps防止除以0

class ResidualConnection(nn.Module):
    def __init__(self, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = LayerNormalization()
        
    def forward(self, x, sublayer):
        """ 
        x是旁路输入, sublayer是直接下游输入
        这里按照原文描述, 先对sublayer执行归一化, 再残差相加
        """
        return x + self.dropout(self.layer_norm(sublayer(x)))

---

## 六、Encoder Block 与 Encoder

In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, self_attn_block: MultiHeadAttentionBlock, ffn_block: FeedForwardBlock, dropout: float):
        super().__init__()
        self.self_attn_block = self_attn_block
        self.ffn_block = ffn_block
        self.residual_connection1, self.residual_connection2 = \
            ResidualConnection(dropout), ResidualConnection(dropout)
        
    def forward(self, x, src_mask):
        """
        由于三个输入都是x自己, 所以是**自注意力**机制
        这里利用lambda表达式表示多头自注意力下游块, 输入三次x自己
        """
        x = self.residual_connection1(x, lambda x: self.self_attn_block(x, x, x, src_mask))
        x = self.residual_connection2(x, lambda x: self.ffn_block(x))

In [ ]:
class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.layer_norm = LayerNormalization()
    
    """
    按照原文经过N个Encoder Block, x的形状始终为(batch, seq_len, d_model)
    最终输出层归一化结果给Decoder
    """
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.layer_norm(x)

---

## 七、Decoder Block 和 Decoder

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(
        self, 
        self_attn_block: MultiHeadAttentionBlock, 
        cross_attn_block: MultiHeadAttentionBlock, 
        ffn_block: FeedForwardBlock, 
        dropout: float
    ):
        super().__init__()
        self.self_attn_block = self_attn_block
        self.cross_attn_block = cross_attn_block
        self.ffn_block = ffn_block
        self.residual_connection1, self.residual_connection2, self.residual_connection3 = \
            ResidualConnection(dropout), ResidualConnection(dropout), ResidualConnection(dropout)
        
    """
    第一个残差连接中的attn与Encoder中的**自注意力**机制相同, 只是前者使用了下三角掩码
    第二个残差连接中的attn使用了**交叉注意力**机制: query来自Decoder本身, 而key和value来自Encoder的输出, 仍使用Encoder的掩码
    第三个残差连接与Encoder中的相同
    """
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connection1(x, lambda x: self.self_attn_block(x, x, x, tgt_mask))
        x = self.residual_connection2(x, lambda x: self.cross_attn_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connection3(x, lambda x: self.ffn_block(x))
        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.layer_norm = LayerNormalization()
        
    """
    与Encoder类似, Decoder也包含多个子层, x的形状始终为(batch, seq_len, d_model)
    Decoder的输入包含了Encoder的输出
    """
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.layer_norm(x)

---

## 八、Projection Linear Layer

In [10]:
class ProjectionLayer(nn.Module):
    """
    定义投影层, 将嵌入向量投影回词表空间
    (batch, seq_len, d_model) -> (batch, seq_len, vocab_size)
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)
        
    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)

---

## 九、Transformer & build_transformer

In [ ]:
class Transformer(nn.Module):
    def __init__(
        self, 
        encoder: Encoder, 
        decoder: Decoder, 
        src_embed: InputEmbedding, 
        tgt_embed: InputEmbedding, 
        src_pos: PositionalEmbedding, 
        tgt_pos: PositionalEmbedding, 
        proj_layer: ProjectionLayer
    ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.proj_layer = proj_layer
        
    def encode(self, src, src_mask):
        src = self.src_embed(src)
        src = self.src_pos(src)
        """
        形状为(batch, seq_len, d_model)
        """
        return self.encoder(src, src_mask)
    
    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
        tgt = self.tgt_embed(tgt)
        tgt = self.tgt_pos(tgt)
        """
        形状为(batch, seq_len, d_model)
        """
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)
    
    def project(self, x):
        return self.proj_layer(x)

In [ ]:
def build_transformer(
    src_vocab_size: int = 1024, 
    tgt_vocab_size: int = 1024, 
    src_seq_len: int = 1024, 
    tgt_seq_len: int = 1024,
    d_model: int = 512,
    N: int = 6,
    num_heads: int = 8,
    dropout: float = 0.1,
    d_ff: int = 2048,
) -> Transformer:
    """
    构建源序列和目标序列的嵌入向量
    """
    src_embed = InputEmbedding(vocab_size=src_vocab_size, d_model=d_model)
    tgt_embed = InputEmbedding(vocab_size=tgt_vocab_size, d_model=d_model)
    
    """
    构建源序列和目标序列**经过位置嵌入**后的向量
    """
    src_pos = PositionalEmbedding(d_model=d_model, seq_len=src_seq_len)
    tgt_pos = PositionalEmbedding(d_model=d_model, seq_len=tgt_seq_len)
    
    """
    分别构建Encoder和Decoder
    分别由 N个EncoderBlock 和 N个DecoderBlock 组成
    """
    encoder_blocks = []
    for _ in range(N):
        encoder_self_attn_block = MultiHeadAttentionBlock(d_model=d_model, num_heads=num_heads, dropout=dropout)
        encoder_ffn_block = FeedForwardBlock(d_model=d_model, d_ff=d_ff, dropout=dropout)
        encoder_block = EncoderBlock(encoder_self_attn_block, encoder_ffn_block, dropout)
        encoder_blocks.append(encoder_block)
        
    decoder_blocks = []
    for _ in range(N):
        decoder_self_attn_block = MultiHeadAttentionBlock(d_model=d_model, num_heads=num_heads, dropout=dropout)
        decoder_cross_attn_block = MultiHeadAttentionBlock(d_model=d_model, num_heads=num_heads, dropout=dropout)
        decoder_ffn_block = FeedForwardBlock(d_model=d_model, d_ff=d_ff, dropout=dropout)
        decoder_block = DecoderBlock(decoder_self_attn_block, decoder_cross_attn_block, decoder_ffn_block, dropout)
        decoder_blocks.append(decoder_block)
        
    encoder = Encoder(layers=nn.ModuleList(encoder_blocks))
    decoder = Decoder(layers=nn.ModuleList(decoder_blocks))
    
    """
    构建ProjectionLayer
    """
    proj_layer = ProjectionLayer(d_model=d_model, vocab_size=tgt_vocab_size)
    
    """
    组合成完整的Transformer
    """
    transformer = Transformer(
        encoder=encoder,
        decoder=decoder,
        src_embed=src_embed,
        tgt_embed=tgt_embed,
        src_pos=src_pos,
        tgt_pos=tgt_pos,
        proj_layer=proj_layer,
    )
    
    """
    构建到最后, 使用xavier_uniform(gain)初始化参数
    参数服从均匀分布U(-a, a), 其中a = gain*sqrt(6/(dim_in+dim_out))
    该初始化方法可保证权重的**输入和输出的分布方差是一致的**, 从而缓解梯度消失或爆炸问题
    """
    for p in transformer.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    
    return transformer

print(f"Total number of parameters is {sum(p.numel() for p in build_transformer().parameters()) / 10**6:.5} M.")

Total number of parameters is 45.682 M.


---

## 十、Tokenizer & Dataset

> 这里以**机器翻译**任务为例, 使用HuggingFace上的wmt数据集

In [13]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import  WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

from pathlib import Path

def get_all_sentences(ds, lang):
    for item in ds:
        yield item["translation"][lang]

"""
定义分词器: 对文本进行分词, 并把每个文本单词映射到单词表中对应的id
本地配置无分词器, 则重新训一个分词器; 本地有分词器, 则直接加载
"""
def get_or_build_tokenzier(config, ds, lang):
    tokenizer_path = Path(config['tokenizer_path'])
    if not Path.exists(tokenizer_path):
        tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], min_frequency=2)
        tokenizer.train_from_iterator(get_all_sentences(ds, lang), trainer=trainer)
        tokenizer.save(str(tokenizer_path))
    else:
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
    return tokenizer

"""
定义训练配置config
"""
def get_config():
    return {
        "batch_size": 8,
        "num_epochs": 20,
        "lr": 10**-4,
        "seq_len": 1,
        "d_model": 512,
        "lang_src": "zh",
        "lang_tgt": "en",
        "preload": None,
        "model_folder": "weights",
        "model_basename": "tmodel",
        "tokenizer_file": "tokenizer_{0}.json",
        "experiment_name": "runs/tmodel"
    }

def get_weights_file_path(config, epoch: int):
    model_folder = config["model_folder"]
    model_basename = config["model_basename"]
    mdoel_filename = f"{model_basename}_{epoch}.pth"
    return str(Path('.') / model_folder / mdoel_filename)

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

"""
构造英译中翻译数据集
重写__len__方法和__getitem__方法
"""
class BilinguialDataset(Dataset):
    def __init__(self, ds, tokenizer_src, tokenizer_tgt, lang_src, lang_tgt, seq_len):
        super().__init__()
        self.ds = ds
        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.lang_src = lang_src
        self.lang_tgt = lang_tgt
        self.seq_len = seq_len
        
        """
        将4个特殊符号转化为tensor id
        """
        self.sos_token = torch.tensor([tokenizer_src.token_to_id('[SOS]')], dtype=torch.int64)
        self.eos_token = torch.tensor([tokenizer_src.token_to_id('[EOS]')], dtype=torch.int64)
        self.pad_token = torch.tensor([tokenizer_src.token_to_id('[PAD]')], dtype=torch.int64)
        
    @staticmethod
    def causal_mask(seq_len):
        """
        创建一个**下三角方阵**, 用于构造因果掩码
        triu函数提取一个方阵的上三角部分(不包括对角线, 即全1), 未被提取的部分(对角线+下方的三角)默认为0
        利用bool表达式获得下三角方阵, 即过滤得到的全0部分, 形状为(1, seq_len, seq_len)
        """
        mask = torch.triu(torch.ones(size=(1, seq_len, seq_len)), diagonal=1).to(torch.int)
        return mask == 0
    
    def __len__(self):
        return len(self.ds)
    
    def __getitem__(self, index):
        src_tgt_pair = self.ds[index]
        text_src = src_tgt_pair[self.lang_src]
        text_tgt = src_tgt_pair[self.lang_tgt]
        
        enc_input_tokens = self.tokenizer_src.encode(text_src).ids
        dec_input_tokens = self.tokenizer_tgt.encode(text_tgt).ids
        
        """
        对当前一条句子做padding
        encoder的输入额外添加sos和eos, 所以需要-2
        decoder的输入只额外添加一个sos, 所以需要-1
        """
        enc_num_padding_tokens = self.seq_len - len(enc_input_tokens) - 2
        dec_num_padding_tokens = self.seq_len - len(dec_input_tokens) - 1
        
        if enc_num_padding_tokens < 0 or dec_num_padding_tokens < 0:
            raise ValueError("Encoder's input or decoder's input is too long.")
        
        """
        <sos>enc_input<eos><pad>
        """
        encoder_input = torch.cat(
            [
                self.sos_token,
                torch.tensor(enc_input_tokens, dtype=torch.int64),
                self.eos_token,
                torch.tensor([self.pad_token] * enc_num_padding_tokens, dtype=torch.int64)
            ],
            dim=0
        )
        
        """
        <sos>dec_input<pad>
        """
        decoder_input = torch.cat(
            [
                self.sos_token,
                torch.tensor(dec_input_tokens, dtype=torch.int64),
                torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64)
            ],
            dim=0
        )
        
        """
        dec_output<eos><pad>
        注意: dec_input有sos无eos, dec_output有eos无sos, 注意比较两者差异
        """
        label = torch.cat(
            [
                torch.tensor(dec_input_tokens, dtype=torch.int64),
                self.eos_token,
                torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64)
            ],
            dim=0
        )
        
        """
        enc_mask: 屏蔽enc_input中的padding
        dec_mask: 屏蔽dec_input中的padding和**后面的单词**
        以下所有的返回值都会扩展一个batch维度
        """
        return {
            "encoder_input": encoder_input, # (seq_len)
            "decoder_input": decoder_input, # (seq_len)
            "encoder_mask": (encoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int(), # (1, 1, seq_len)
            "decoder_mask": (decoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int() \
                            & BilinguialDataset.causal_mask(decoder_input.shape[0]), # (1, 1, seq_len) & (1, seq_len, seq_len) -> (1, seq_len, seq_len)
            "label":    label,              # (seq_len)
            "text_src": text_src,           # (seq_len)
            "text_tgt": text_tgt,           # (seq_len)
        }
        
def get_dataset(config):
    ds_raw = load_dataset('wmt/wmt19', f'{config['lang_src']}-{config['lang_tgt']}', split='train')
    tokenizer_src = get_or_build_tokenzier(config=config, ds=ds_raw, lang=config['lang_src'])
    tokenizer_tgt = get_or_build_tokenzier(config=config, ds=ds_raw, lang=config['lang_tgt'])
    
    train_ds_size = int(0.8 * len(ds_raw))
    valid_ds_size = len(ds_raw) - train_ds_size
    train_ds_raw, valid_ds_raw = random_split(ds_raw, [train_ds_size, valid_ds_size]) # 随机划分数据集
    
    train_ds = BilinguialDataset(
                    ds=train_ds_raw, #
                    tokenizer_src=tokenizer_src,
                    tokenizer_tgt=tokenizer_tgt,
                    lang_src=config['lang_src'],
                    lang_tgt=config['lang_tgt'],
                    seq_len=config['seq_len'],
                )
    valid_ds = BilinguialDataset(
                    ds=valid_ds_raw, #
                    tokenizer_src=tokenizer_src,
                    tokenizer_tgt=tokenizer_tgt,
                    lang_src=config['lang_src'],
                    lang_tgt=config['lang_tgt'],
                    seq_len=config['seq_len'],
                )
    
    max_len_src, max_len_tgt = 0, 0
    
    for item in ds_raw:
        ids_src = tokenizer_src.encode(item[config['lang_src']]).ids
        ids_tgt = tokenizer_tgt.encode(item[config['lang_tgt']]).ids
        max_len_src = max(max_len_src, len(ids_src))
        max_len_tgt = max(max_len_tgt, len(ids_tgt))
        
    train_dataloader = DataLoader(
        train_ds,
        batch_size=config['batch_size'],
        shuffle=True,
    )
    valid_dataloader = DataLoader(
        valid_ds,
        batch_size=1,
        shuffle=True,
    )
    
    return train_dataloader, valid_dataloader, tokenizer_src, tokenizer_tgt

def get_model(config, vocab_size_src, vocab_size_tgt):
    return build_transformer(
        src_vocab_size=vocab_size_src,
        tgt_vocab_size=vocab_size_tgt,
        src_seq_len=config['seq_len'],
        tgt_seq_len=config['seq_len'],
        d_model=config['d_model'],
    )

---

## 十一、Train & Validation

In [ ]:
"""
训练代码
"""
def train_model(config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    Path(config["model_folder"]).mkdir(parents=True, exist_ok=True)
    
    train_dataloader, valid_dataloader, tokenizier_src, tokenizer_tgt = get_dataset(config=config)
    model = get_model(config=config, vocab_size_src=tokenizier_src.get_vocab_size(), vocab_size_tgt=tokenizer_tgt.get_vocab_size())
    
    """
    使用Adam优化器
    """
    optimizer = torch.optim.Adam(params=model.parameters(), lr=config["lr"], eps=1e-9)
    
    """
    支持预加载和断点续炼
    """
    initial_epoch, global_step = 0, 0
    if config["preload"]:
        model_filename = get_weights_file_path(config=config, epoch=config["preload"])
        print(f"Preloading model {model_filename}")
        state = torch.load(model_filename)
        initial_epoch = state["epoch"] + 1
        optimizer.load_state_dict(state["optimizer_state_dict"])
        global_step = state["global_step"]
    
    """
    使用交叉熵损失函数
    注意不让pad token参与loss的计算
    """
    loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizier_src.token_to_id("[PAD]"), label_smoothing=0.1).to(device)
    
    from tqdm import tqdm
    for epoch in range(initial_epoch, config["num_epochs"]):
        model.train()
        batch_iterator = tqdm(train_dataloader, desc=f"Processing Epoch {epoch:02d}")
        
        for batch in batch_iterator:
            """
            获得一个批次内的输入信息
            """
            encoder_input = batch["encoder_input"].to(device) # (batch, seq_len)
            decoder_input = batch["decoder_input"].to(device) # (batch, seq_len)
            encoder_mask = batch["encoder_mask"].to(device)   # (batch, 1, 1, seq_len), 只屏蔽pad
            decoder_mask = batch["decoder_mask"].to(device)   # (batch, 1, seq_len, seq_len), 同时屏蔽pad和未来信息
            
            """
            输入transformer, 得到输出并计算loss
            """
            encoder_output = model.encode(encoder_input, encoder_mask)                                # (batch, seq_len, d_model)
            decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)  # (batch, seq_len, d_model)
            proj_output = model.project(decoder_output)                                               # (batch, seq_len, tgt_vocab_size), 记录每个位置的独热单词id
            
            label = batch["label"].to(device)                                                         # (batch, seq_len), 记录每个位置的词表id
            
            """
            CELoss的输入格式: (N, C) <--> (N,)
            其中N表示批次内的单词总数, C表示类别数(词表大小), 符合调用规范
            """
            loss = loss_fn(proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))
            batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})
            
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
            
            global_step += 1
            
        """
        每个epoch结束时保存状态
        """
        model_filename = get_weights_file_path(config=config, epoch=f"{epoch:02d}")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": loss,
            "global_step": global_step
        }, model_filename)

In [ ]:
"""
推理代码
"""
def greedy_decode(model, src, mask_src, tokenizer_src, tokenizer_tgt, max_seq_len, device):
    sos_id = tokenizer_tgt.token_to_id("[SOS]")
    eos_id = tokenizer_tgt.token_to_id("[EOS]")
    
    """
    计算一次encoder_output, 供后续decoder循环推理下一个单词
    因此可先将encoder_output预先计算出来, 并初始化decoder_input的sos
    decoder_input拓展第0个维度, 以适应模型**批处理输入**
    """
    encoder_output = model.encode(src, mask_src)
    decoder_input = torch.tensor(sos_id, dtype=torch.int64).view(-1, 1).to(device)
    
    while True:
        if decoder_input.shape[1] == max_seq_len:
            break  # 限制最大生成长度 
        
        """
        基于当前的decoder_input, 计算下三角mask和下一步decoder_output
        decoder_output的形状: (1, cur_seq_len, d_model), 这里简单推演一下预测到第二个token的过程:
            1. decoder_input: (1, 2, d_model), 其seq_len为2, 远小于encoder_input的seq_len
            2. 经过decoder_masked_attn: (1, 2, d_model), causal_mask只和当前seq_len有关
            3. 经过decoder_cross_attn: (1, 2, d_model)
                3.1. 乘以W: query_from_decoder: (1, 2, d_model); key/value_from_encoder: (1, seq_len, d_model)
                3.2. 算attn_score: (1, n_heads, 2, d_k) @ (1, n_heads, d_k, seq_len) -> (1, n_heads, 2, seq_len) (长方形)
                     再乘以多头value: (1, n_heads, 2, seq_len) @ (1, n_heads, seq_len, d_k) -> (1, n_heads, 2, d_k)
                3.3. 多头拼接, 再乘以Wo: (1, 2, d_model)
            4. 经过decoder_ffn: (1, 2, d_model)
            5. 取最后预测的token经过decoder_proj: (1, 1, vocab_size), 这里直接取最大值对应索引id即可
        """
        mask_dec = BilinguialDataset.causal_mask(decoder_input.shape[1]).to(device)
        decoder_output = model.decode(encoder_output, mask_src, decoder_input, mask_dec)
        
        """
        将当前最后一个token投影回词表空间: (1, 1, d_model) -> (1, 1, vocab_size)
        取出输出中的**最大概率**, 将其作为当前步的的预测结果
        auto-regressive: 将当前预测单词id拼回decoder_input中, 继续预测下一个单词
        """
        prob = model.project(decoder_output[:, -1])
        _, next_word_id = torch.max(prob, dim=-1)
        decoder_input = torch.cat([decoder_input, next_word_id.view(-1, 1).to(device)], dim=1)
    
        if next_word_id == eos_id:
            break  # 预测到eos单词就停止生成
        
    return decoder_input.squeeze(0) # 重新挤掉batch维度
        

def validate(
    model, 
    valid_ds, 
    tokenizer_src, 
    tokenizer_tgt, 
    max_seq_len, 
    device, 
):
    model.eval()
    
    from tqdm import tqdm
    with torch.no_grad():
        for batch in valid_ds:
            encoder_input = batch["encoder_input"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)
            
            assert encoder_input.shape[0] == 1, "Batch size must be 1 for inference."
            
            """
            自回归贪婪输出
            将输出结果从vocab空间decode回真正的文本token
            """
            model_output = greedy_decode(
                                model=model, 
                                encoder_input=encoder_input, 
                                encoder_mask=encoder_mask,
                                tokenizer_src=tokenizer_src,
                                tokenizer_tgt=tokenizer_tgt,
                                max_seq_len=max_seq_len,
                                device=device
                            )
            model_output_text = tokenizer_tgt.decode(model_output.detach().cpu().numpy())
            tqdm.write(f"{model_output_text}")  # tqdm进度条运行时用这个方法打印比较美观